<font color="red" size="10">
<b>DEMO TIME: 10 mins</b>
</font>

# About this Notebook
## Additional Histories in ART

This notebook demonstrates how to use **additional histories** for complex agent training scenarios in ART (Agent Reinforcement Training).

Additional histories allow you to include multiple separate conversation histories within a single trajectory, enabling training of agents with:
- Non-linear conversation flows
- Agents that call sub-agents
- History compaction and summarization
- Preservation of special tokens across multi-turn conversations


This notebook is for the *Demo* for Session 3 for Develop Self-Improving AI Agents with Reinforcement Learning Live Event with O'Reilly Media by
[Nicole Koenigstein](https://www.linkedin.com/in/nicole-koenigstein/).


# Additional Resources


<a target="_blank" href="https://learning.oreilly.com/library/view/ai-agents-the/0642572247775/">
  <img src="https://img.shields.io/badge/AI%20Agents%20Book-Read%20on%20O'Reilly-d40101?style=flat" alt="AI Agents Book – Read on O'Reilly"/>
</a>


In [ ]:
import warnings
warnings.filterwarnings('ignore')  # Suppress all warnings

warnings.warn("This warning will be hidden")
print("Script continues...")

In [1]:
import codecs
from IPython.display import Markdown, display

def reveal_answer(which: str = "", key: str = ""):
    answers = {
        "q1": "Vs gur pung grzcyngr fgevcf fcrpvny gbxraf sebz rneyvre gheaf, gura gurfr gbxraf arire rkvfg va gur genvavat qngn. Gur zbqry pnaabg trg perqvg nffvtazrag sbe gur ernfbavat cnggrea, fb EY bayl yrneaf sebz gur svany bhgchg. Ol fgbevat rnpu gheavat uvfgbel frcnengryl va nqqvgvbany uvfgbevrf, lbh cerfreir gur gbxraf naq tvir EY n pyrne fvtany nobhg jung ernfbavat fgrcf jbex.",
        "q2": "Vs lbh bayl genva ba gur znva pbairefngvba, gura gur zbqry frrf gur bhgchg ohg abg gur fhontnag fgrcf gung cebqhprq vg. Gung oernxf perqvg nffvtazrag orpnhfr gur cbfg ivpgbel erjneq pnaabg or nggevohgrq gb gur evtug fhontnag npgvbaf naq genwrgbevrf. Ol fgbevat rnpu fhontnag pbairefngvba nf n nqqvgvbany uvfgbel, lbh tvir EY gur shapgvbaf, npgvbaf, naq vagrezrqvngr fvgnnyf vg arrqf gb yrnea ubj gb qryrtngr, ubj gb irevsl, naq ubj gb pbzcbfr erfhygf."
    }

    if key.strip().lower() == "decode":
        secret = answers.get(which, "")
        decoded = codecs.decode(secret, "rot_13")
        display(Markdown("### **Decoded Answer**\n\n<font size='5' color='green'>" + decoded + "</font>"))
    else:
        display(Markdown("### **Answer remains encoded.**\n\nProvide key = 'decode' to reveal."))

In [2]:
# @title Installation
# Portions adapted from Unsloth Notebooks (https://github.com/unslothai/notebooks)
# Copyright (c) Unsloth contributors.
# License: GNU LGPL v3.0.
# Modifications by OpenPipe:
# - switched to uv
# - changed vllm/triton pinning logic
# - added protobuf pins
# - adjusted syntax for pushing to HF
# See /licenses/LGPL-3.0.txt and /licenses/GPL-3.0.txt for full text.

%%capture
import os

if "COLAB_" not in "".join(os.environ.keys()):
    !uv pip install openpipe-art[backend]==0.4.11 tenacity "mcp>=1.11.0" "gql<4" aiohttp --prerelease allow --no-cache-dir
else:
    try:
        import numpy

        get_numpy = f"numpy=={numpy.__version__}"
    except:
        get_numpy = "numpy"
    try:
        import subprocess

        is_t4 = "Tesla T4" in str(subprocess.check_output(["nvidia-smi"]))
    except:
        is_t4 = False
    get_vllm, get_triton = (
        ("vllm==0.9.2", "triton==3.2.0") if is_t4 else ("vllm", "triton")
    )
    !uv pip install --upgrade \
        openpipe-art[backend]==0.4.11 tenacity pillow==11.3.0 protobuf==5.29.5 {get_vllm} {get_numpy} --prerelease allow --no-cache-dir
    !uv pip install -qqq {get_triton}

In [3]:
import art
from art.trajectories import Trajectory, History, Choice
import art
from art.local import LocalBackend
from art.utils.logging import info, ok, step



/usr/local/lib/python3.12/dist-packages/pydantic/_internal/_generate_schema.py:2249: UnsupportedFieldAttributeWarning: The 'repr' attribute with value False was provided to the `Field()` function, which has no effect in the context it was used. 'repr' is field-specific metadata, and can only be attached to a model field using `Annotated` metadata or by assignment. This may have happened because an `Annotated` type alias using the `type` statement was used, or if the `Field()` function was attached to a single member of a union type.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/pydantic/_internal/_generate_schema.py:2249: UnsupportedFieldAttributeWarning: The 'frozen' attribute with value True was provided to the `Field()` function, which has no effect in the context it was used. 'frozen' is field-specific metadata, and can only be attached to a model field using `Annotated` metadata or by assignment. This may have happened because an `Annotated` type alias using the `type` 

## What are Additional Histories?

In ART, a trajectory typically contains a single sequence of messages. However, some advanced use cases require training on multiple related but separate conversations within the same trajectory context.

Each trajectory can contain:
- A **primary** `messages_and_choices` sequence (the main conversation)
- An optional list of **additional_histories**, where each history contains its own `messages_and_choices` and optional `tools`


# Question 1

<font color="red" size="10">
<b>Question to ALL:</b>
</font>
<br><br>
<font color="blue" size="5">
<b>If the model’s chat template strips special tokens like <code>&lt;think&gt;</code>
from earlier turns, why does that matter for RL training, and how do
<code>additional_histories</code> fix it? </b>
</font>

In [4]:
#reveal_answer("q1", "decode")

## Example 1: Preserving Special Tokens in Multi-Turn Conversations

Some models, like Qwen 3, use chat templates that remove special tokens (such as `<think>`) from previous turns in multi-turn conversations. This can interfere with training when you want the model to learn from its thinking process across all turns.

By splitting each turn into a separate history, you can preserve these tokens for training.


In [5]:
from art.trajectories import Trajectory, History, Choice
from openai.types.chat import ChatCompletionMessage

labels = {"task": "arithmetic", "demo": "additional_histories"}  # your workshop labels

trajectory_preserve_tokens = Trajectory(
    messages_and_choices=[
        {"role": "user", "content": "What is 2+2?"},
        Choice(
            index=0,
            finish_reason="stop",
            message=ChatCompletionMessage(
                role="assistant",
                content="<think>I need to add 2 and 2</think>4",
            ),
        ),
    ],
    additional_histories=[
        History(
            messages_and_choices=[
                {"role": "user", "content": "What is 2+2?"},
                Choice(
                    index=0,
                    finish_reason="stop",
                    message=ChatCompletionMessage(role="assistant", content="4"),
                ),
                {"role": "user", "content": "What is 3+3?"},
                Choice(
                    index=0,
                    finish_reason="stop",
                    message=ChatCompletionMessage(
                        role="assistant",
                        content="<think>I need to add 3 and 3</think>6",
                    ),
                ),
            ]
        )
    ],
    reward=0.9,
    metrics={
        "preserved_reasoning_tokens": True,
        "num_turns": 2,
    },
)

print("Labels for demo:", labels)


Labels for demo: {'task': 'arithmetic', 'demo': 'additional_histories'}


# Question 2

<font color="red" size="10">
<b>Question to ALL:</b>
</font>
<br><br>
<font color="blue" size="5">
<b>
In a coordinator agent that delegates to sub agents, what goes wrong in RL if you only train on the main conversation, and why is it useful to store each sub agent dialogue as an additional history? </b>
</font>

In [6]:
reveal_answer("q2", "decode")


### **Decoded Answer**

<font size='5' color='green'>If you only train on the main conversation, then the model sees the output but not the subagant steps that produced it. That breaks credit assignment because the post victory reward cannot be attributed to the right subagant actions and trajetories. By storing each subagant conversation as a additional history, you give RL the functions, actions, and intermediate sitaals it needs to learn how to delegate, how to verify, and how to compose results.</font>

## Example 2: Training Agents That Call Sub-Agents

When an agent delegates work to sub-agents, each sub-agent conversation can be stored as an additional history. This allows the model to learn from both the main conversation flow and the sub-agent interactions.


In [7]:
from art.trajectories import Trajectory, History, Choice
from openai.types.chat import ChatCompletionMessage

ANALYZE_ID = "call_analyze_1"
FIX_ID = "call_fix_1"

trajectory_sub_agents = Trajectory(
    messages_and_choices=[
        {"role": "user", "content": "Analyze this codebase and fix any bugs"},

        # Assistant tool call: analyze_code
        Choice(
            index=0,
            finish_reason="tool_calls",
            message=ChatCompletionMessage(
                role="assistant",
                content=None,
                tool_calls=[
                    {
                        "id": ANALYZE_ID,
                        "type": "function",
                        "function": {
                            "name": "analyze_code",
                            "arguments": '{"file": "main.py", "task": "Find potential bugs"}',
                        },
                    }
                ],
            ),
        ),

        # Tool result for analyze_code
        {
            "role": "tool",
            "tool_call_id": ANALYZE_ID,
            "content": (
                "Found 3 potential issues: null pointer on line 42, "
                "memory leak on line 87, uninitialized variable on line 120"
            ),
        },

        # Assistant tool call: fix_issues
        Choice(
            index=0,
            finish_reason="tool_calls",
            message=ChatCompletionMessage(
                role="assistant",
                content=None,
                tool_calls=[
                    {
                        "id": FIX_ID,
                        "type": "function",
                        "function": {
                            "name": "fix_issues",
                            "arguments": '{"issue": "null pointer on line 42", "fix": "Add null check"}',
                        },
                    }
                ],
            ),
        ),

        # Tool result for fix_issues
        {
            "role": "tool",
            "tool_call_id": FIX_ID,
            "content": "Fixed by adding null check: if (ptr != nullptr) { ... }",
        },

        # Final assistant answer
        Choice(
            index=0,
            finish_reason="stop",
            message=ChatCompletionMessage(
                role="assistant",
                content=(
                    "I've analyzed the codebase and fixed the critical null pointer issue on line 42. "
                    "The fix adds a null check before dereferencing the pointer."
                ),
            ),
        ),
    ],
    additional_histories=[
        # Sub agent 1
        History(
            messages_and_choices=[
                {"role": "system", "content": "You are a code analysis expert. Analyze code for potential bugs."},
                {"role": "user", "content": "Find potential bugs in main.py"},
                Choice(
                    index=0,
                    finish_reason="stop",
                    message=ChatCompletionMessage(
                        role="assistant",
                        content=(
                            "Found 3 potential issues:\n"
                            "1. Null pointer on line 42\n"
                            "2. Memory leak on line 87\n"
                            "3. Uninitialized variable on line 120"
                        ),
                    ),
                ),
            ]
        ),
        # Sub agent 2
        History(
            messages_and_choices=[
                {"role": "system", "content": "You are a bug fixing expert. Fix code issues safely and correctly."},
                {"role": "user", "content": "Fix the null pointer issue on line 42"},
                Choice(
                    index=0,
                    finish_reason="stop",
                    message=ChatCompletionMessage(
                        role="assistant",
                        content=(
                            "Fixed by adding null check:\n"
                            "```cpp\n"
                            "if (ptr != nullptr) {\n"
                            "    // safe to use ptr\n"
                            "}\n"
                            "```"
                        ),
                    ),
                ),
            ]
        ),
    ],
    reward=0.85,
    metrics={  # numeric or bool only
        "sub_agents_used": 2,
        "bugs_found": 3,
        "bugs_fixed": 1,
        "task_completed": True,
    },
)

labels = {"task": "code_analysis_and_fix"}  # keep strings out of metrics

print("Example 2: Sub-Agents")
print("=" * 60)
print("Primary history messages:", len(trajectory_sub_agents.messages_and_choices))
print("Additional histories:", len(trajectory_sub_agents.additional_histories))
print("Reward:", trajectory_sub_agents.reward)
print("Metrics:", trajectory_sub_agents.metrics)
print("Labels:", labels)


Example 2: Sub-Agents
Primary history messages: 6
Additional histories: 2
Reward: 0.85
Metrics: {'sub_agents_used': 2, 'bugs_found': 3, 'bugs_fixed': 1, 'task_completed': True}
Labels: {'task': 'code_analysis_and_fix'}


In [8]:
from art.trajectories import Trajectory, History, Choice
from openai.types.chat import ChatCompletionMessage

trajectory_compaction = Trajectory(
    messages_and_choices=[
        {"role": "system", "content": "You are a helpful assistant."},
        {
            "role": "user",
            "content": (
                "Compacted conversation history: The user asked about quantum entanglement, and I explained the "
                "basic concept of how particles can be correlated even when separated."
            ),
        },
        {"role": "user", "content": "Tell me more about the history of quantum entanglement"},
        Choice(
            index=0,
            finish_reason="stop",
            message=ChatCompletionMessage(
                role="assistant",
                content=(
                    "Quantum entanglement was first proposed by Einstein, Podolsky, and Rosen in 1935 in what's now "
                    "called the EPR paradox. They were concerned that quantum mechanics seemed to violate locality, "
                    "the idea that objects can only be influenced by their immediate surroundings. However, "
                    "experiments by John Bell in the 1960s and subsequent tests have confirmed that entanglement is "
                    "a real physical phenomenon. Today, entanglement is a key resource in quantum computing and "
                    "quantum cryptography."
                ),
            ),
        ),
        {"role": "user", "content": "How is it used in quantum computing?"},
        Choice(
            index=0,
            finish_reason="stop",
            message=ChatCompletionMessage(
                role="assistant",
                content=(
                    "In quantum computing, entanglement is used to create quantum gates that operate on multiple "
                    "qubits simultaneously. For example, the CNOT gate creates entanglement between two qubits. "
                    "This enables quantum algorithms to process information in ways that classical computers cannot. "
                    "Some algorithms show large speedups for specific tasks, such as factoring (Shor's algorithm) or "
                    "search (Grover's algorithm)."
                ),
            ),
        ),
    ],
    additional_histories=[
        History(
            messages_and_choices=[
                {"role": "system", "content": "You are a helpful assistant."},
                {"role": "user", "content": "Explain quantum entanglement"},
                Choice(
                    index=0,
                    finish_reason="stop",
                    message=ChatCompletionMessage(
                        role="assistant",
                        content=(
                            "Quantum entanglement is a phenomenon where two or more particles become correlated so "
                            "that the quantum state of each particle cannot be described independently, even when the "
                            "particles are separated by large distances. Measuring one particle lets you predict "
                            "properties of the other, regardless of distance. This 'spooky action at a distance' "
                            "(as Einstein called it) has been experimentally verified and is a fundamental aspect of "
                            "quantum mechanics."
                        ),
                    ),
                ),
                {"role": "user", "content": "Can you give me a simple example?"},
                Choice(
                    index=0,
                    finish_reason="stop",
                    message=ChatCompletionMessage(
                        role="assistant",
                        content=(
                            "Imagine two coins that are perfectly anti-correlated. If one is heads, the other is "
                            "tails, every time. If you separate them far apart, observing one instantly tells you "
                            "what you'd observe for the other. This is only an analogy: real entanglement is not "
                            "about hidden predetermined coin states, but it helps build intuition."
                        ),
                    ),
                ),
            ]
        )
    ],
    reward=0.92,
    metrics={
        # numeric or bool only
        "history_compacted": True,
        "original_turns": 4,
        "compacted_turns": 3,
    },
)

labels = {"task": "quantum_physics_qa"}  # keep string labels outside metrics

print("Example 3: History Compaction")
print("=" * 60)
print("Primary history messages:", len(trajectory_compaction.messages_and_choices))
print("Additional histories:", len(trajectory_compaction.additional_histories))
print("Reward:", trajectory_compaction.reward)
print("Metrics:", trajectory_compaction.metrics)
print("Labels:", labels)

print("\nCompacted history (primary):")
for msg in trajectory_compaction.messages_and_choices[:3]:
    print(f"  {msg['role']}: {msg['content'][:120]}...")

print("\nOriginal history (additional):")
for msg in trajectory_compaction.additional_histories[0].messages_and_choices:
    if isinstance(msg, dict):
        print(f"  {msg['role']}: {msg['content'][:120]}...")
    else:
        # msg is a Choice
        content = (msg.message.content or "")[:120]
        print(f"  Choice: {content}...")


Example 3: History Compaction
Primary history messages: 6
Additional histories: 1
Reward: 0.92
Metrics: {'history_compacted': True, 'original_turns': 4, 'compacted_turns': 3}
Labels: {'task': 'quantum_physics_qa'}

Compacted history (primary):
  system: You are a helpful assistant....
  user: Compacted conversation history: The user asked about quantum entanglement, and I explained the basic concept of how part...
  user: Tell me more about the history of quantum entanglement...

Original history (additional):
  system: You are a helpful assistant....
  user: Explain quantum entanglement...
  Choice: Quantum entanglement is a phenomenon where two or more particles become correlated so that the quantum state of each par...
  user: Can you give me a simple example?...
  Choice: Imagine two coins that are perfectly anti-correlated. If one is heads, the other is tails, every time. If you separate t...


## Example 4: Complete Implementation with Tools

This example shows how to create a trajectory with additional histories that include tool definitions, demonstrating a more complete setup.


In [9]:
from art.trajectories import Trajectory, History, Choice
from openai.types.chat import ChatCompletionMessage

# Define tools for the main trajectory
main_tools = [
    {
        "type": "function",
        "function": {
            "name": "delegate_to_sub_agent",
            "description": "Delegate a task to a specialized sub-agent",
            "parameters": {
                "type": "object",
                "properties": {
                    "sub_agent_type": {
                        "type": "string",
                        "description": "Type of sub-agent (e.g., 'researcher', 'writer', 'analyst')",
                    },
                    "task": {"type": "string", "description": "Task to delegate"},
                },
                "required": ["sub_agent_type", "task"],
            },
        },
    }
]

# Define tools for sub-agent histories
sub_agent_tools = [
    {
        "type": "function",
        "function": {
            "name": "search",
            "description": "Search for information",
            "parameters": {
                "type": "object",
                "properties": {"query": {"type": "string", "description": "Search query"}},
                "required": ["query"],
            },
        },
    }
]

# Stable ids for tool calls so tool responses can reference them
RESEARCH_DELEGATE_ID = "call_research_1"
WRITER_DELEGATE_ID = "call_writer_1"
SEARCH_ID = "call_search_1"

trajectory_with_tools = Trajectory(
    messages_and_choices=[
        {
            "role": "system",
            "content": "You are a coordinator agent that delegates tasks to specialized sub-agents.",
        },
        {"role": "user", "content": "Research the latest developments in AI safety and write a summary"},

        # Delegate to researcher
        Choice(
            index=0,
            finish_reason="tool_calls",
            message=ChatCompletionMessage(
                role="assistant",
                content=None,
                tool_calls=[
                    {
                        "id": RESEARCH_DELEGATE_ID,
                        "type": "function",
                        "function": {
                            "name": "delegate_to_sub_agent",
                            "arguments": (
                                '{"sub_agent_type": "researcher", '
                                '"task": "Research latest AI safety developments"}'
                            ),
                        },
                    }
                ],
            ),
        ),
        {
            "role": "tool",
            "tool_call_id": RESEARCH_DELEGATE_ID,
            "content": "Research completed: Found recent papers on AI alignment, interpretability, and robustness from 2024.",
        },

        # Delegate to writer
        Choice(
            index=0,
            finish_reason="tool_calls",
            message=ChatCompletionMessage(
                role="assistant",
                content=None,
                tool_calls=[
                    {
                        "id": WRITER_DELEGATE_ID,
                        "type": "function",
                        "function": {
                            "name": "delegate_to_sub_agent",
                            "arguments": (
                                '{"sub_agent_type": "writer", '
                                '"task": "Write summary of AI safety research"}'
                            ),
                        },
                    }
                ],
            ),
        ),
        {
            "role": "tool",
            "tool_call_id": WRITER_DELEGATE_ID,
            "content": "Summary written: Recent AI safety developments focus on alignment, interpretability, and robustness.",
        },

        # Final assistant message
        Choice(
            index=0,
            finish_reason="stop",
            message=ChatCompletionMessage(
                role="assistant",
                content=(
                    "I've coordinated the research and writing tasks. "
                    "The summary of recent AI safety developments is ready."
                ),
            ),
        ),
    ],
    tools=main_tools,
    additional_histories=[
        # Researcher sub-agent
        History(
            messages_and_choices=[
                {
                    "role": "system",
                    "content": "You are a research specialist. Search for and gather information on given topics.",
                },
                {"role": "user", "content": "Research latest AI safety developments"},

                # Researcher uses search tool
                Choice(
                    index=0,
                    finish_reason="tool_calls",
                    message=ChatCompletionMessage(
                        role="assistant",
                        content=None,
                        tool_calls=[
                            {
                                "id": SEARCH_ID,
                                "type": "function",
                                "function": {
                                    "name": "search",
                                    "arguments": '{"query": "AI safety developments 2024"}',
                                },
                            }
                        ],
                    ),
                ),
                {
                    "role": "tool",
                    "tool_call_id": SEARCH_ID,
                    "content": "Found recent papers on AI alignment, interpretability, and robustness from 2024.",
                },

                # Researcher final response
                Choice(
                    index=0,
                    finish_reason="stop",
                    message=ChatCompletionMessage(
                        role="assistant",
                        content=(
                            "Research completed: Found recent papers on AI alignment, interpretability, "
                            "and robustness from 2024."
                        ),
                    ),
                ),
            ],
            tools=sub_agent_tools,
        ),

        # Writer sub-agent
        History(
            messages_and_choices=[
                {"role": "system", "content": "You are a writing specialist. Create clear and concise summaries."},
                {"role": "user", "content": "Write summary of AI safety research"},
                Choice(
                    index=0,
                    finish_reason="stop",
                    message=ChatCompletionMessage(
                        role="assistant",
                        content=(
                            "Summary: Recent AI safety developments emphasize alignment (keeping AI goals consistent "
                            "with human intent), interpretability (understanding model decisions), and robustness "
                            "(reliable behavior under distribution shift and adversarial pressure)."
                        ),
                    ),
                ),
            ]
        ),
    ],
    reward=0.88,
    metrics={
        # numeric or bool only
        "sub_agents_used": 2,
        "tools_used": 3,
        "task_completed": True,
    },
)

labels = {"task": "research_and_summarize"}  # keep strings out of metrics

print("Example 4: Complete Implementation with Tools")
print("=" * 60)
print("Primary history messages:", len(trajectory_with_tools.messages_and_choices))
print("Additional histories:", len(trajectory_with_tools.additional_histories))
print("Main trajectory tools:", len(trajectory_with_tools.tools) if trajectory_with_tools.tools else 0)
print(
    "Sub-agent 1 tools:",
    len(trajectory_with_tools.additional_histories[0].tools)
    if trajectory_with_tools.additional_histories[0].tools
    else 0,
)
print(
    "Sub-agent 2 tools:",
    len(trajectory_with_tools.additional_histories[1].tools)
    if trajectory_with_tools.additional_histories[1].tools
    else 0,
)
print("Reward:", trajectory_with_tools.reward)
print("Metrics:", trajectory_with_tools.metrics)
print("Labels:", labels)


Example 4: Complete Implementation with Tools
Primary history messages: 7
Additional histories: 2
Main trajectory tools: 1
Sub-agent 1 tools: 1
Sub-agent 2 tools: 0
Reward: 0.88
Metrics: {'sub_agents_used': 2, 'tools_used': 3, 'task_completed': True}
Labels: {'task': 'research_and_summarize'}


## How Tokenization Works

When a trajectory with additional histories is tokenized:
1. The main history (from `messages_and_choices`) is tokenized first
2. Each additional history is tokenized separately
3. The training weight is distributed across all tokenized results
4. Each history maintains its own context and token boundaries


In [10]:
# Demonstrate the structure of trajectories with additional histories
print("Trajectory Structure Overview")
print("=" * 60)

all_trajectories = [
    ("Preserve Tokens", trajectory_preserve_tokens),
    ("Sub-Agents", trajectory_sub_agents),
    ("History Compaction", trajectory_compaction),
    ("With Tools", trajectory_with_tools)
]

for name, traj in all_trajectories:
    print(f"\n{name}:")
    print(f"  Primary messages: {len(traj.messages_and_choices)}")
    print(f"  Additional histories: {len(traj.additional_histories)}")
    print(f"  Reward: {traj.reward}")
    if traj.tools:
        print(f"  Tools: {len(traj.tools)}")
    for i, hist in enumerate(traj.additional_histories, 1):
        print(f"    History {i}: {len(hist.messages_and_choices)} messages")
        if hist.tools:
            print(f"      Tools: {len(hist.tools)}")


Trajectory Structure Overview

Preserve Tokens:
  Primary messages: 2
  Additional histories: 1
  Reward: 0.9
    History 1: 4 messages

Sub-Agents:
  Primary messages: 6
  Additional histories: 2
  Reward: 0.85
    History 1: 3 messages
    History 2: 3 messages

History Compaction:
  Primary messages: 6
  Additional histories: 1
  Reward: 0.92
    History 1: 5 messages

With Tools:
  Primary messages: 7
  Additional histories: 2
  Reward: 0.88
  Tools: 1
    History 1: 5 messages
      Tools: 1
    History 2: 3 messages


## Data Structure Reference

The `History` and `Trajectory` classes support additional histories as follows:

```python
@dataclass
class History:
    messages_and_choices: list[dict[str, Any]]
    tools: list[Tool] | None = None

@dataclass
class Trajectory:
    messages_and_choices: list[dict[str, Any]]
    tools: list[Tool] | None = None
    additional_histories: list[History] = field(default_factory=list)
    reward: float | None = None
    metrics: dict[str, Any] = field(default_factory=dict)
```


In [11]:
from art.trajectories import Trajectory, History

print("Trajectory Fields:")
print("=" * 60)
for name, f in Trajectory.model_fields.items():
    ann = getattr(f, "annotation", None)
    print(f"  {name}: {ann}")

print("\nHistory Fields:")
print("=" * 60)
for name, f in History.model_fields.items():
    ann = getattr(f, "annotation", None)
    print(f"  {name}: {ann}")


Trajectory Fields:
  messages_and_choices: list[typing.Union[openai.types.chat.chat_completion_developer_message_param.ChatCompletionDeveloperMessageParam, openai.types.chat.chat_completion_system_message_param.ChatCompletionSystemMessageParam, openai.types.chat.chat_completion_user_message_param.ChatCompletionUserMessageParam, openai.types.chat.chat_completion_assistant_message_param.ChatCompletionAssistantMessageParam, openai.types.chat.chat_completion_tool_message_param.ChatCompletionToolMessageParam, openai.types.chat.chat_completion_function_message_param.ChatCompletionFunctionMessageParam, openai.types.chat.chat_completion.Choice]]
  tools: list[openai.types.chat.chat_completion_tool_param.ChatCompletionToolParam] | None
  additional_histories: list[art.trajectories.History]
  reward: <class 'float'>
  metrics: dict[str, float | int | bool]
  metadata: dict[str, float | int | str | bool | None]
  logs: list[str]
  start_time: <class 'datetime.datetime'>

History Fields:
  message

In [12]:
from art.trajectories import Trajectory, History

def describe_model(model_cls):
    for name, f in model_cls.model_fields.items():
        ann = getattr(f, "annotation", None)
        required = f.is_required()
        default = None if required else f.default
        print(f"  {name}")
        print(f"    type: {ann}")
        print(f"    required: {required}")
        if not required:
            print(f"    default: {default}")

print("Trajectory Schema:")
print("=" * 60)
describe_model(Trajectory)

print("\nHistory Schema:")
print("=" * 60)
describe_model(History)


Trajectory Schema:
  messages_and_choices
    type: list[typing.Union[openai.types.chat.chat_completion_developer_message_param.ChatCompletionDeveloperMessageParam, openai.types.chat.chat_completion_system_message_param.ChatCompletionSystemMessageParam, openai.types.chat.chat_completion_user_message_param.ChatCompletionUserMessageParam, openai.types.chat.chat_completion_assistant_message_param.ChatCompletionAssistantMessageParam, openai.types.chat.chat_completion_tool_message_param.ChatCompletionToolMessageParam, openai.types.chat.chat_completion_function_message_param.ChatCompletionFunctionMessageParam, openai.types.chat.chat_completion.Choice]]
    required: True
  tools
    type: list[openai.types.chat.chat_completion_tool_param.ChatCompletionToolParam] | None
    required: False
    default: None
  additional_histories
    type: list[art.trajectories.History]
    required: False
    default: []
  reward
    type: <class 'float'>
    required: True
  metrics
    type: dict[str, floa

## Summary

Additional histories enable powerful training scenarios:

1. **Preserving Special Tokens**: Keep reasoning tokens intact across multi-turn conversations
2. **Sub-Agent Training**: Train agents that delegate to specialized sub-agents
3. **History Compaction**: Train on both compacted and original conversation histories
4. **Complex Workflows**: Model hierarchical agent interactions and multi-stage processes

When tokenized, each history is processed independently, and training weights are distributed across all histories in the trajectory.

These trajectories can be used for training with ART's training pipeline.
Each trajectory includes:
  - Primary conversation history
  - Additional histories for context
  - Reward signals
  - Metadata and metrics


In [13]:
trajectory_set = [
    trajectory_preserve_tokens,
    trajectory_sub_agents,
    trajectory_compaction,
    trajectory_with_tools,
]

print(f"Collected {len(trajectory_set)} trajectories for optimization")


Collected 4 trajectories for optimization


In [14]:
for i, traj in enumerate(trajectory_set, 1):
    print(f"Trajectory {i}: reward={traj.reward}")


Trajectory 1: reward=0.9
Trajectory 2: reward=0.85
Trajectory 3: reward=0.92
Trajectory 4: reward=0.88


In [15]:
bad_trajectory = Trajectory(
    messages_and_choices=[
        {"role": "user", "content": "What is 2+2?"},
        Choice(
            index=0,
            finish_reason="stop",
            message=ChatCompletionMessage(
                role="assistant",
                content="I am not sure. Maybe 5?",
            ),
        ),
    ],
    reward=0.1,
)
